In [ ]:
# Topic-RAG+  (This script includes chunking strategy to efficiently handle long documents and complex queries that would require to retrieve large number of documents.)

In [3]:
import pandas
import pandas as pd
import collections
import time
import faiss
import numpy
import pickle
import swifter
from sklearn.metrics.pairwise import cosine_similarity

from langchain_experimental.text_splitter import SemanticChunker
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer , util
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

In [2]:
!python --version

Python 3.10.14


In [3]:
import bertopic
bertopic.__version__

'0.16.3'

In [4]:
data_df= pd.read_pickle("your_data.pkl")

In [6]:
embeddings = data_df["sentence_embeddings"].to_list()
contents = data_df["content_translated_processed_regex_cleaned"].to_list()
years = data_df["year"].to_list()
print(len(embeddings), len(contents),len(years))

4711 4711 4711


In [7]:
contents, embeddings , years = numpy.array(contents), numpy.array(embeddings)
print(contents.shape, embeddings.shape)

(4711,) (4711, 768) (4711,)


#### Topic Modeling BERTOPIC

In [11]:
# Extract vocab to be used in BERTopic
vocab_counter = collections.Counter()
tokenizer = CountVectorizer().build_tokenizer()

In [12]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, random_state=42, metric="cosine", verbose=True)

In [13]:

from bertopic.vectorizers import ClassTfidfTransformer

# Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True,gen_min_span_tree=True)


vectorizer_model = CountVectorizer(stop_words="english",ngram_range=(1,1))

#  Create topic representation
ctfidf_model = ClassTfidfTransformer()

In [ ]:
from sentence_transformers import SentenceTransformer 
embedding_model = SentenceTransformer("jinaai/jina-embeddings-v2-base-en", trust_remote_code=True)

In [9]:
# KeyBERT
keybert_model = KeyBERTInspired()

# Part-of-Speech
# pos_model = PartOfSpeech("en_core_web_sm")

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    # "OpenAI": openai_model,  # Uncomment if you will use OpenAI
    "MMR": mmr_model,
    # "POS": pos_model
}

In [ ]:
start_time = time.time()

print("Start",start_time)

topic_model = BERTopic(
  # Pipeline models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  ctfidf_model=ctfidf_model, 
  representation_model=representation_model,
  # Hyperparameters
  top_n_words=10,
  verbose=True, calculate_probabilities=True,low_memory=True
)
topics,probs=topic_model.fit_transform(contents, embeddings)

end_time = time.time()
print("End",end_time)
execution_time = end_time - start_time

print("Execution time:", execution_time, "seconds")

In [ ]:
import numpy as np

# `probs` is the topic probabilities array
np.save("../models/probs_RAGG.npy", probs)


In [ ]:
document_info_topic_model= topic_model.get_document_info(contents)
document_info_topic_model.head()

In [ ]:
topic_model.save("your_model_path.pkl",serialization="pickle",save_ctfidf=True, save_embedding_model=embedding_model)

In [8]:
from bertopic import BERTopic
bertopic_models= BERTopic.load("your_model_path.pkl")
# bertopic_models.get_topic_info()

Fri Dec 20 15:13:35 2024 Building and compiling search function


In [ ]:
topic_freq = bertopic_models.get_topic_freq()


# Filter out the topic with ID -1, which typically represents outliers or noise
filtered_topic_freq = topic_freq[topic_freq['Topic'] != -1]


topic_freq_list = [
    f"Topic {topic_id} has {frequency} documents"
    for topic_id, frequency in filtered_topic_freq[['Topic', 'Count']].values
]

# Display the list of topic frequencies with added text
for entry in topic_freq_list:
    print(entry)

In [57]:
top_n = 10  # Adjust N as needed
top_topics = topic_freq[topic_freq.Topic != -1].sort_values('Count', ascending=False).head(top_n)

In [ ]:
# you can check the top N topics along with this document count using this plot.

import matplotlib.pyplot as plt

# Step 1: Create a mapping of topic ID to topic name
topic_names = {
    topic_id: ", ".join([word[0] for word in bertopic_models.get_topic(topic_id)[:3]])  # Top 3 terms
    for topic_id in top_topics['Topic']
}

# Step 2: Map topic names to the 'Topic' column
top_topics['Topic Name'] = top_topics['Topic'].map(topic_names)

# Step 3: Plot with topic names as x-axis labels
plt.figure(figsize=(8,4))
bars = plt.bar(top_topics['Topic Name'], top_topics['Count'], color='purple')

# Add the document count above each bar with a small gap
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 13, str(int(yval)), ha='center', fontsize=8)

plt.xlabel('Topic Name', fontsize=8)
plt.ylabel('Document Count', fontsize=8)
plt.title('Top 10 Topics by Document Count', fontsize=10)
plt.xticks(rotation=30, ha='right', fontsize=8)  # Rotate labels for better readability
plt.yticks(fontsize=8)
plt.tight_layout()  # Adjust layout to fit rotated labels

# Save the figure
plt.savefig("../Figures/topic_distribution", dpi=600)
plt.show()


In [ ]:
# Extract topic labels and top 10 terms
topic_labels = [
    f"Topic {topic_id}: " + ", ".join([word[0] for word in bertopic_models.get_topic(topic_id)[:10]])
    for topic_id in top_topics['Topic']
]
print("Top 10 topics")
topic_labels

In [ ]:
hierarchy_fig= bertopic_models.visualize_hierarchy()
hierarchy_fig.save

In [ ]:
document_infor_topic_model= bertopic_models.get_document_info(contents)
document_infor_topic_model.head()

In [ ]:
# merging columns from original dataframe to document-topic mapping df .
data_to_merge=data_df[['uid','year','sentence_embeddings']]

merged_df = pd.concat([data_to_merge.reset_index(drop=True), document_infor_topic_model.reset_index(drop=True),],axis=1)

merged_df.head()

In [ ]:
topic_embeddings=bertopic_models.topic_embeddings_
topic_embeddings
# print(type(topic_embeddings))

# Creating Vector index

In [12]:
embedding_dimension = embedding_model.get_sentence_embedding_dimension()
print(f"Embedding Dimension: {embedding_dimension}")

Embedding Dimension: 768


In [13]:
original_docs= merged_df['Document'].to_list()
len(original_docs)

4711

- Topic-Document Index - Created Topic-Document mapping. We arrange documents based on their topics. (separate index for each topic)
-Adding Chunking strategy before creating Document FAISS index

In [26]:
!pip install --quiet langchain_experimental langchain_openai

In [20]:
EMBEDDINGS_PATH = "jinaai/jina-embeddings-v2-base-en" # both MPS and GPU support!

In [21]:
# semantic chunking strategy

from typing import List

class CustomEmbeddings:
        
        def __init__(self, model):
            self.model = SentenceTransformer(model, trust_remote_code=True)
    
        def embed_documents(self, texts: List[str]) -> List[List[float]]:
            return [self.model.encode(t).tolist() for t in texts]
        
        def embed_query(self, query: str) -> List[float]:
            return self.model.encode([query])

text_splitter = SemanticChunker(CustomEmbeddings(EMBEDDINGS_PATH), breakpoint_threshold_type="percentile")

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
import pickle

embedding_model = embedding_model # jina AI model

# Initialize containers for topic indices and metadata
doc_indices = {} # is to store document embeddings topic wise
doc_topic_metadata = {}

# Process each topic
for topic in sorted(merged_df["Topic"].unique()):
    # Filter documents for the current topic
    topic_docs = merged_df[merged_df["Topic"] == topic]

    # Initialize FAISS index for this topic
    embedding_dim = embedding_model.get_sentence_embedding_dimension()
    chunk_doc_index = faiss.IndexFlatL2(embedding_dim)  # L2 similarity for FAISS
    metadata_store = []  # To store metadata for each chunk

    # Process each document within the topic
    for _, row in topic_docs.iterrows():
        document, uid = row["Document"], row["uid"]
        
         # Remove newlines from document content if necessary
        document = document.replace('\n', ' ').replace('\r', ' ').strip()  # Clean up newlines

        # Use create_documents function to split text into chunks using semantic chunker
        documents = text_splitter.create_documents([document])

        # Generate embeddings for each chunk
        embeddings = np.array([embedding_model.encode(doc.page_content) for doc in documents])

        # Add embeddings to FAISS index
        chunk_doc_index.add(embeddings.astype(np.float32))  # Ensure correct dtype for FAISS
        
        

        # Store metadata for each chunk (document metadata + chunk index)
        for chunk_idx, doc in enumerate(documents):
            metadata_store.append({
                "Topic": topic,
                "uid": uid,
                "chunk_id": chunk_idx,
                "chunk_doc": doc.page_content,
                "chunk_embeddings": embeddings[chunk_idx].astype(np.float32)
            })

    # Save the FAISS index and metadata for the topic
    doc_indices[topic] = chunk_doc_index
    doc_topic_metadata[topic] = metadata_store
print("Semantic chunking with FAISS indexing and metadata tracking complete!")

In [ ]:
print("--------------Saving Document FAISS index for each topic------------------")
# Save FAISS indices for each topic
for topic, index in doc_indices.items():
    print(f"Topic {topic} index: {index}")
    faiss.write_index(index, f'faiss_index_chunked/document_index_topic_{topic}.faiss')  # Save the FAISS index to a file
print("Document FAISS indices saved for each topic!")

print("-----Saving Metadata-----------------------")

with open("chunked_docs_metadata.pkl", "wb") as f:
    pickle.dump(doc_topic_metadata, f)
print("Metadata saved using pickle!")


print("------------------------------------------------------------------------------")

In [ ]:
doc_topic_metadata_flat = []
for md in doc_topic_metadata:
    mdv = doc_topic_metadata[md]
    doc_topic_metadata_flat += mdv
print("Length of metadata: ", len(doc_topic_metadata_flat))
doc_topic_metadata_flat = pd.DataFrame(doc_topic_metadata_flat)
doc_topic_metadata_flat.head()

In [50]:
doc_topic_metadata_flat['chunk_embeddings'][0].shape

(768,)

In [53]:
doc_topic_metadata_flat.to_pickle("chunked_docs_metadata_df.pkl") # save the metadata for future reference.

In [ ]:
metadata_df=pd.read_pickle("chunked_docs_metadata_df.pkl")
metadata_df.head()

In [55]:
import swifter
# Function to calculate token length (rough approximation by splitting by space)
def calculate_token_length(text):
    return len(text.split())  # Split by spaces and count the tokens (words)

metadata_df['chunk_token_length'] = metadata_df['chunk_doc'].swifter.apply(calculate_token_length)
metadata_df.head(10)
metadata_df.to_pickle("chunked_docs_metadata_df.pkl")

Pandas Apply:   0%|          | 0/10794 [00:00<?, ?it/s]

In [ ]:
import os
import faiss
import re

# Directory where the FAISS indices are stored
index_directory = "faiss_index_chunked"

# Dictionary to store loaded FAISS indices
loaded_document_indices = {}

def extract_number(filename):
    # Use regex to find a number in the filename, including negative numbers
    match = re.search(r'-?\d+', filename)  # Matches optional '-' followed by digits
    if match:
        return int(match.group())
    return float('inf')  # Return a very large number if no valid number
# Read all FAISS files in the directory, sort them by extracted number
sorted_files = sorted(
    [file for file in os.listdir(index_directory) if file.endswith(".faiss")],
    key=extract_number
)

# Load each FAISS index
for file_name in sorted_files:
    file_path = os.path.join(index_directory, file_name)  # Full path to the index file
    topic_number = extract_number(file_name)  # Extract the topic number
    try:
        # Load the FAISS index
        index = faiss.read_index(file_path)
        # Store in the dictionary
        loaded_document_indices[topic_number] = index
        print(f"Loaded FAISS index for topic: {topic_number}, Index size: {index.ntotal}")
    except Exception as e:
        print(f"Failed to load index from {file_name}: {e}")


In [16]:
loaded_document_indices

{-1: <faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x3fea6cbd0> >,
 0: <faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x365809380> >,
 1: <faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x365863420> >,
 2: <faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x34daad290> >,
 3: <faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x34daade30> >,
 4: <faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x401bffb10> >,
 5: <faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x365548930> >,
 6: <faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x401a960a0> >,
 7: <faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x401aae5b0> >,
 8: <faiss.swigfaiss.IndexFlatL2; proxy of <S

In [18]:
new_doc_indices=loaded_document_indices.copy()

# Initialize LLAMA 3.1 8b model - Generator Model to generate responses.

In [19]:
LLAMA_CPP_PATH = "../models/Meta-Llama-3.1-8B-Instruct-Q5_K_M.gguf" # from local directory
TOKENIZER_PATH= "meta-llama/Meta-Llama-3.1-8B-Instruct" # from Huggingface or local cache3

In [20]:
from transformers import AutoTokenizer
config = AutoTokenizer.from_pretrained(TOKENIZER_PATH, use_fast=True)
print(f"Maximum sequence length: {config.model_max_length}")

Maximum sequence length: 131072


In [21]:
from llama_cpp import Llama
from langchain_community.llms import LlamaCpp
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])

print("LOADING LLAMA.CPP MODEL...")
llama_model = Llama(
    model_path=LLAMA_CPP_PATH,
    n_gpu_layers=-1, # -1 put all models layers on the GPU
    max_tokens=8192, # output: chunk plus instructions (plus buffer for Llama-3 special tokens)
    n_ctx=100000, # input: chunk plus explanations (plus buffer for Llama-3 special tokens)
    f16_kv=True, # False means higher (=32bit) precision for key/value cache
    callback_manager=callback_manager,
    seed=42,
    #chat_format="chatml",
    verbose=False,
)
print("LLAMA.CPP MODEL LOADED.")

print("PREPARING LLAMA.CPP MODEL...")
from accelerate import Accelerator
accelerator = Accelerator()
llama_model = accelerator.prepare(llama_model)
print("LLAMA.CPP MODEL PREPARED.")

print("LOADING TOKENIZER...")
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("TOKENIZER LOADED.")

import torch
if torch.backends.mps.is_available():
    print("DEVICE:", round(torch.mps.driver_allocated_memory()/(1024.**3), 3), "GB ALLOCATED ON MPS")
elif torch.cuda.is_available():
    print("DEVICE:", round(torch.cuda.max_memory_reserved() /(1024.**3), 3), "GB ALLOCATED ON CUDA")    


LOADING LLAMA.CPP MODEL...
LLAMA.CPP MODEL LOADED.
PREPARING LLAMA.CPP MODEL...
LLAMA.CPP MODEL PREPARED.
LOADING TOKENIZER...
TOKENIZER LOADED.
DEVICE: 31.874 GB ALLOCATED ON MPS


# Multiple Query


- Topic-RAG+ (Our Method)

- Generator Function ( generator prompt and getAnswer)
    -"generator_prompt" function - a prompt for the generator model to follow this instruction and generate a response in this format.
    - "getAnswer" function - to generate a response based on the instruction and input text.


In [26]:
type(metadata_df['chunk_embeddings'])

pandas.core.series.Series

# Historian Questions - Qualitative evaluation (Case study 4.7)


In [ ]:
import numpy as np
import datetime


def generator_prompt(instruction, input=None, uid=None, summary_size='concise'):
    return (
        "You are a skilled summarizer. Your task is to generate a {summary_size} summary of the provided context, "
        f"using the relevant document with UID {uid}. Focus solely on the information directly relevant to the instruction, ignoring irrelevant content. "
        "Ensure the summary is clear, concise, and presented in a logical order, avoiding any repetition of sentences and editorial comments. "
        "Do not repeat sentences to fill space; instead, provide a meaningful summary that aligns with the specified size.Provide a narrative summary that includes" "only the relevant information. Emphasize clarity and conciseness while covering all key points."
        "Avoid restating or repeating content, and ensure diversity in phrasing. Do not include any statements about the writing process in the beginning or end of" "the summary?.\n\n"
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{input}\n\n"
        "### Generated Response:\n"

    ).format(summary_size=summary_size)

def getAnswer(INSTRUCTION, MAX_TOKENS=1000):

    combined_mapping = []
    print("\n")
    for topic in results["topic_info"]:
        print(f"Topic ID: {topic['topic_id']}, Probability: {topic['probability']}")
     # print(f"Retrieved Documents: {topic['retrieved_documents']}")


        selected_docs = topic['selected_documents']  # selected_docs = [(uid1, sim_score1), (uid2, sim_score2), ...]
        selected_doc_contents = topic['selected_doc_contents']
        chunk_doc_index= topic['chunk_id']

        # Create a mapping for each selected document
        for (uid, score), content , chunk_id in zip(selected_docs, selected_doc_contents,chunk_doc_index):
            combined_mapping.append({
                "uid": uid,
                "score": score,
                "content": content,
                "topic_id": topic['topic_id'],
                "chunk_id": chunk_id,
            })
        for topic in results["topic_info"]:
            user_input = topic["user_input_value"]


        print("\n")
        print(f"Selected Documents based on user input value '{(user_input)}' are as follows :")
        print("\n")
            # combined_mapping contains all selected documents with their UIDs, scores, topic IDs and content
        for item in combined_mapping:
            print(f"UID: {item['uid']}, Score: {item['score']}, Topic ID: {item['topic_id']}, Chunk ID: {item['chunk_id']} , Content: {item['content']},\n")

    # Combine contents from selected documents in the combined mapping
    selected_doc_contents = [item['content'] for item in combined_mapping]  # Get the contents
    selected_doc_uid = [item['uid'] for item in combined_mapping]  # Get the UIDs
    input_text = '\n'.join(selected_doc_contents) # Combine the contents

    prompt = generator_prompt(INSTRUCTION, input=input_text , uid=selected_doc_uid,summary_size='concise')
    # inputs = tokenizer(prompt, return_tensors="pt")
    answer = llama_model(prompt, max_tokens=MAX_TOKENS,seed=42) # To generate response (according to the prompt) using llama model initialized above.
    # answer = model.generate(inputs.input_ids, max_new_tokens=MAX_TOKENS)

    print("\n-----------------------------------------------------------\n")
    print("Total number of retrieved documents",len(selected_doc_contents))
    print("\n-----------------------------------------------------------\n")

    print ("Original Documents",selected_doc_contents)
    print("\n---------------------------------\n")
    print("\n---------------------------------\n")


    return answer['choices'][0]['text']



topic_ids = metadata_df['Topic'].unique()

print("Total number of topics:", len(topic_ids))
print(f"Original topic IDs: {topic_ids}:")


def query_rag_system(query,embedding_model, topic_model, new_doc_indices, docs_df):


    # Step 1: Embed the query
    query_embedding = embedding_model.encode([query])
    query_embedding = query_embedding.astype(np.float32)


    # Step 2: Apply BERTopic modeling to get topic representations of the query
    topics, probs = topic_model.transform([query], query_embedding)  # Transform the query to get its topic

    print("Topics for the input query assigned by bertopic model :", topics)
    print("Probabilities for the input query assigned by bertopic model :", probs)

    # Find the highest probability
    highest_probability = np.max(probs[0])

    # Set the threshold as 80% of the highest probability
    threshold = highest_probability * 0.5

    #>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>.
    ignored_topics = [i for i in range(len(probs[0])) if probs[0][i] < threshold]
    # Print the ignored topics along with their probability values
    print("Ignored topics and their probabilities:")
    for topic in ignored_topics:
     print(f"Topic {topic}: Probability {probs[0][topic]:.4f}")

     #>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>.

    # Get indices of topics that meet or exceed the threshold
    filtered_topics = [i for i in range(len(probs[0])) if probs[0][i] >= threshold]

    print(f"Filtered topics: {filtered_topics}")

    # Sort filtered topics based on probabilities
    k_topics=len(filtered_topics)
    top_query_topics_indices = sorted(filtered_topics, key=lambda idx: probs[0][idx], reverse=True)[:k_topics]



    # Check if the query maps to an existing topic
    if topics[0] != -1:
        print(f"The query maps to topic {topics[0]} with probability {probs[0][top_query_topics_indices]}")
    else:
        print("The query did not map to any existing topic and was considered an outlier.")

    print("Query Topic assigned by BERTopic model:", topics[0])
    print("Query Topic list based on Probs (list if more than one topic contributes to the input query ) :", topics) # list if more than one topic

    query_topic_ids = top_query_topics_indices
    print ("Documents to be retrieved based on the query topic ids:",query_topic_ids)
    top_probabilities = probs[0][top_query_topics_indices]
    print("Top probabilities:", top_probabilities)
    print("\n")

    topic_info = []

    for index, topic_idx in enumerate(query_topic_ids):

        print(f"................................................Topic ID: {topic_idx} ................................................")

        #...............................................
        relevant_topic_id = int(topic_idx)
        print(f"Retrieving documents for topic ID {relevant_topic_id}")
        print(f"Processing Topic {index + 1}/{len(query_topic_ids)}: Topic ID {relevant_topic_id}")

        # topic_info = []
        chunk_id={}
        retrieved_documents = {}
        retrieved_document_embeddings = {}
        retrieved_doc_contents = {}
        retrieved_doc_uid = {}
        selected_docs = {}
        selected_doc_contents = {}

        # Retrieve the top k documents for this topic using the corresponding FAISS index
        try:
            k_docss = new_doc_indices[relevant_topic_id].ntotal  # Total number of documents in the index
            print(f"Total number of documents in the index for topic ID {relevant_topic_id}: {k_docss}")
            distances, indices = new_doc_indices[relevant_topic_id].search(query_embedding, k_docss)  # Using FAISS to get the top k_docs
            retrieved_documents[relevant_topic_id] = indices[0].tolist()

            # print(f"Initially Retrieved Documents for topic ID {relevant_topic_id}: {retrieved_documents[relevant_topic_id]}")

            # Filter docs_df to include only the rows corresponding to the relevant topic
            filtered_docs_df = docs_df[docs_df['Topic'] == relevant_topic_id]
            # Check the size of the filtered DataFrame (number of rows and columns)
            num_rows, num_columns = filtered_docs_df.shape
            print(f"Filtered DataFrame size: {num_rows} rows and {num_columns} columns")

            # Optionally, you can just check the number of rows (documents)
            print(f"Number of documents in the filtered DataFrame for topic {relevant_topic_id}: {num_rows}")

            print ("Dataframe columns :",filtered_docs_df.columns)

            chunk_id[relevant_topic_id]=filtered_docs_df['chunk_id']

            retrieved_document_embeddings[relevant_topic_id] = filtered_docs_df['chunk_embeddings'].tolist()
            retrieved_doc_contents[relevant_topic_id] = filtered_docs_df['chunk_doc'].tolist()
            retrieved_doc_uid[relevant_topic_id] = filtered_docs_df['uid'].tolist()

            # # Access document embeddings and contents using the indices from docs_df
            # retrieved_document_embeddings[relevant_topic_id] = docs_df.iloc[retrieved_documents[relevant_topic_id]]['sentence_embeddings'].tolist()
            # retrieved_doc_contents[relevant_topic_id] = docs_df.iloc[retrieved_documents[relevant_topic_id]]['Document'].tolist()
            # retrieved_doc_uid[relevant_topic_id] = docs_df.iloc[retrieved_documents[relevant_topic_id]]['uid'].tolist()

            print("\n")
            # Check the size of the retrieved document embeddings
            print(f"Size of retrieved_document_embeddings for Topic ID {relevant_topic_id}: {len(retrieved_document_embeddings[relevant_topic_id])}")


            # Calculate cosine similarity between the query embedding and retrieved document embeddings
            cosine_similarities = cosine_similarity(query_embedding.reshape(1, -1), np.array(retrieved_document_embeddings[relevant_topic_id])).flatten()

            # Create a list of tuples (document_uid, cosine_similarity)
            document_scores = [
                (retrieved_doc_uid[relevant_topic_id][i], cosine_similarities[i])
                for i in range(len(cosine_similarities))
            ]
            # Sort the documents by cosine similarity in descending order
            sorted_document_scores = sorted(document_scores, key=lambda x: x[1], reverse=True)

            # print(f"\nCosine Similarity Scores for Topic ID {relevant_topic_id} as follows : \n ")
            #
            # for rank, (doc_uid, sim) in enumerate(sorted_document_scores):
            #     print(f"Rank {rank + 1}: Document UID: {doc_uid}, Cosine Similarity: {sim}")

            #..............................
            # Prepare the top 25 cosine similarities for the input prompt
            top_n = 10
            top_scores = sorted_document_scores[:top_n]  # Get the top 10 scores
            top_scores_str = "\n".join([f"Rank {rank + 1}: UID: {doc_uid}, Similarity: {sim:.4f}"
                                         for rank, (doc_uid, sim) in enumerate(top_scores)])

             # Ask the user how many documents to retrieve from the same cluster (here it is topic id )
            user_input = int(input(f"How many documents would you like to select in topic {relevant_topic_id} "
                                   f"for further processing from the {len(cosine_similarities)} documents?\n Query :{[query]} \n\n"
                                   f"Top {top_n} Cosine Similarities:\n{top_scores_str}\n"))

            # Select the top user_input documents based on sorted cosine similarities
            selected_docs[relevant_topic_id] = sorted_document_scores[:user_input]
            print("\n")
            # Print the selected documents with their topic information
            print(f"Selected Documents with Topic Information for Topic ID {relevant_topic_id}: {selected_docs[relevant_topic_id]}")

            # Get the actual content of the selected documents
            selected_doc_contents[relevant_topic_id] = [retrieved_doc_contents[relevant_topic_id][
                retrieved_doc_uid[relevant_topic_id].index(doc_uid)
            ] for doc_uid, _ in selected_docs[relevant_topic_id]]

            # print(f"Selected Document Contents for Topic ID {relevant_topic_id}: {selected_doc_contents[relevant_topic_id]}")

            topic_details = {
                "topic_id": relevant_topic_id,
                "chunk_id":chunk_id[relevant_topic_id],
                "probability": probs[0][relevant_topic_id] if relevant_topic_id < len(probs[0]) else None,
                "retrieved_documents": retrieved_documents.get(relevant_topic_id, []),
                "retrieved_embeddings": retrieved_document_embeddings.get(relevant_topic_id, []),
                "retrieved_doc_contents": retrieved_doc_contents.get(relevant_topic_id, []),
                "retrieved_doc_uid": retrieved_doc_uid.get(relevant_topic_id, []),
                "selected_documents": selected_docs[relevant_topic_id],
                "selected_doc_contents": selected_doc_contents[relevant_topic_id],
                "user_input_value": user_input,
            }
            topic_info.append(topic_details)

        except KeyError:
            print(f"Topic ID {relevant_topic_id} not found in new_doc_indices.")

    return {
        "query": query,
        "query_embedding": query_embedding,
        "topic_info": topic_info
    }

start_time = datetime.datetime.now()



queries =["Which arguments did proponents and opponents of the 1979 Volksbegehren put forward?", "Which articles take an explicit position on favour or against the Volksbegehren?" , "Give me a name list of all scientific experts mentioned in the articles in the context of the debate about the Kaiseraugst nuclear plant and in how many article they are mentioned. Then for each expert quoted, give a summary of the statement and classify the thematic the person spoke about.", "Give an overview of the year in which environmental topics were first discussed in the different newspapers. As output produce a table with the following columns: newspaper title, date of publication, article title, 10 word article summary " , " How did articles create a sense of fear in their readers? Use quotes from the articles to exemplify your answer." ,"Which articles strongly advocate nuclear power. What are the arguments they put forward and Which articles strongly oppose nuclear power. What are the arguments they put forward? Do arguments change over time", " Give me 10 examples of articles which either subtly argue in favour or against nuclear power but appear to be neutral at first sight?" , "Which experts are linked to the nuclear waste topic?","Describe the ratio of articles concerned with swiss national politics compared to international politics?"]


all_results_marten_qa=[]
# -------------------------------------Retrieval------------------------------------------------------------------
for query in queries:
    retrieval_start_time = datetime.datetime.now()

    # Retrieval based on topic representation of the query and then by query embedding.
    results = query_rag_system(query, embedding_model, bertopic_models, new_doc_indices,metadata_df)

    retrieval_end_time = datetime.datetime.now()

    elapsed_time_retrieval = retrieval_end_time - retrieval_start_time


    print ("........................Retrieval completed..................... \n ")

    print("Elapsed time for retrieval",elapsed_time_retrieval)

    print("----------------------------------------------------------------------------------")

    # -------------------------------------Generation------------------------------------------------------------------
    print("\n")
    print( "........................Generating Response - Starting......................")

    generation_time_start = datetime.datetime.now()

    response = getAnswer(INSTRUCTION=f"Explain the details of {query}")
    print("__________________________________Generated Response:________________________--")
    print(response)
    generation_time_end = datetime.datetime.now()

    generated_response_time = generation_time_end - generation_time_start

    print("Elapsed time for generation",generated_response_time)

     # Store the result along with the response
    all_results_marten_qa.append({
        "query": results["query"],
        "topic_info": results["topic_info"],
        "response": response
    })

    print("******************************************************")

end_time =datetime.datetime.now()

Total_time = end_time - start_time

print("Total Elapsed time ( both retrieval and generation) ",Total_time)

In [ ]:
print("...............................OUTPUT.Historian questions...............................................")
for result in all_results_marten_qa:
    print("Query:", result["query"])

    for topic in result["topic_info"]:
        print("\n")
        print(f"Topic ID: {topic['topic_id']}, Probability: {topic['probability']}")
        print("------")
        # print(f"Selected Documents: {topic['selected_documents']}")
        # print("------")
        # print(f"Selected Document Contents: {topic['selected_doc_contents']}")
        # print("-----")
        print("Response:", result["response"])
    print("******************************************************")
    print("\n")

In [ ]:
output_file = "../RAG_results/output_historian_queries_RAG.txt"

# Open the file in write mode
with open(output_file, "w") as file:
    file.write("...............................OUTPUT.Historian questions...............................................\n")
    for result in all_results_marten_qa:
        file.write(f"Query: {result['query']}\n")
        print("\n")

        topic_ids = [topic['topic_id'] for topic in result["topic_info"]]
        file.write("------\n")

        # Write query-specific topic IDs at the beginning
        file.write(f"All Topic IDs for this query: {', '.join(map(str, sorted(topic_ids)))}\n")
        for topic in result["topic_info"]:
            file.write("\n")
            file.write(f"Topic ID: {topic['topic_id']}, Probability: {topic['probability']}\n")
            file.write("------\n")
            file.write(f"No.of Retrieved docs based on User Input value:{topic['user_input_value']}\n")
            file.write("-------\n")
            # Uncomment below lines if you want to include additional information
            # file.write(f"Selected Documents: {topic['selected_documents']}\n")
            # file.write("------\n")
            # file.write(f"Selected Document Contents: {topic['selected_doc_contents']}\n")
            # file.write("-----\n")
            file.write(f"Response: {result['response']}\n")
        file.write("******************************************************\n")
        file.write("\n")

print(f"Output successfully saved to {output_file}")
